# Lab 07: Retrieval, Tools, and Agent Interfaces

Modern AI systems are not just models — they are **systems built around models** that interact with knowledge and environments. In this lab, you will progressively build such a system by adding components around a language model:

1. **Direct answering**: the model answers using parametric knowledge
2. **Retrieval-Augmented Generation (RAG)**: retrieve documents from a local corpus
3. **Tool calling**: connect the model to external tools (calculator, internet search)
4. **Function calling**: structured tool invocation with Pydantic schemas
5. **Agent routing**: decide when to answer directly, retrieve, or call a tool
6. **Trace logging**: log and visualize agent behavior
7. **Tool description experiments**: modify tool descriptions and observe routing changes
8. **Evaluation**: measure agent routing accuracy

The key insight: **the same model becomes dramatically more capable when wrapped in a system that connects it to knowledge and tools.**

---

In [ ]:
# =============================================================================
# Google Colab Setup: Download lab materials
# =============================================================================
# Run this cell first to download the required files (docs/, traces/, etc.)

import os

if not os.path.exists("docs"):
    !wget -q https://github.com/Kevin-Miao/cdss94_labs/releases/download/lab/lab_07.zip
    !unzip -q lab_07.zip
    !rm lab_07.zip
    print("Lab materials downloaded and extracted.")
else:
    print("Lab materials already present.")

## Section 0: Setup with OpenRouter

We will use [OpenRouter](https://openrouter.ai) to access language models. OpenRouter provides a unified API that is compatible with the OpenAI SDK, so you can use the familiar `openai` Python package.

### 0.1 Create an OpenRouter Account

1. Go to [https://openrouter.ai](https://openrouter.ai) and sign up (Google/GitHub login works).
2. Navigate to [https://openrouter.ai/keys](https://openrouter.ai/keys) and click **Create Key**.
3. Copy your API key. It starts with `sk-or-v1-`.

### 0.2 Set Your API Key

Set your API key as an environment variable **before** launching the notebook:

```bash
export OPENROUTER_API_KEY="sk-or-v1-your-key-here"
```

Alternatively, you can paste it directly in the cell below (not recommended for shared environments).

In [ ]:
# =============================================================================
# Install dependencies (run once)
# =============================================================================

!pip install openai sentence-transformers faiss-cpu pydantic duckduckgo-search

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

import os
import json
import re
import time
from openai import OpenAI
from pathlib import Path

# ---------------------------------------------------------------------------
# OpenRouter setup
# ---------------------------------------------------------------------------
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "Please set your OPENROUTER_API_KEY environment variable.\n"
        "  export OPENROUTER_API_KEY='sk-or-v1-...'\n"
        "Or set it directly: OPENROUTER_API_KEY = 'sk-or-v1-...'"
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Model to use: a capable but affordable model
MODEL = "google/gemini-2.0-flash-001"

print(f"OpenRouter client ready.")
print(f"Model: {MODEL}")

In [ ]:
# =============================================================================
# PROVIDED: Helper function to call the model
# =============================================================================

def call_model(prompt, system_prompt=None, temperature=0.0, max_tokens=512):
    """Call a language model via OpenRouter.

    Args:
        prompt: The user message.
        system_prompt: Optional system message.
        temperature: Sampling temperature (0 = deterministic).
        max_tokens: Maximum tokens to generate.

    Returns:
        dict with keys:
            'content': the model's response text
            'prompt_tokens': number of input tokens
            'completion_tokens': number of output tokens
            'total_tokens': total tokens used
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    usage = response.usage
    return {
        "content": response.choices[0].message.content,
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "total_tokens": usage.total_tokens if usage else 0,
    }


print("call_model() helper defined.")

In [ ]:
# =============================================================================
# Test your API connection
# =============================================================================

test_response = call_model("What is 2 + 2? Reply with just the number.")
print(f"Response: {test_response['content']}")
print(f"Tokens used: {test_response['total_tokens']}")
print("\nAPI connection working!")

---

## Section 1: Direct Question Answering

Our first approach is the simplest: give the model a question and let it answer using only its **parametric knowledge** (what it learned during training). No external documents, no tools, no search.

This establishes our **baseline** for comparison with more sophisticated methods.

```
user query --> model --> answer
```

**TODO 1.1:** Write a function `ask_direct(query)` that:
1. Sends the query to the model with no additional context
2. Returns a dict with the model's response and token usage

In [ ]:
# TODO 1.1: Implement direct question answering (~5-10 lines)
#
# Hint: Use call_model() with a simple system prompt like
#   "Answer the user's question concisely."
#
# Return a dict with keys:
#   'response': str (the model's answer)
#   'total_tokens': int

def ask_direct(query):
    """Answer a question using only the model's parametric knowledge."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
test = ask_direct("What is 2 + 2?")
print(f"Response: {test['response']}")
print(f"Tokens: {test['total_tokens']}")

**TODO 1.2:** Test on sample questions. These questions are chosen to highlight different model capabilities and limitations:
- Knowledge questions (may or may not be in training data)
- Math questions (models struggle with precise computation)
- Factual recall questions (specific dates/facts)

In [ ]:
# TODO 1.2: Test direct answering on these sample questions (~10-15 lines)
#
# For each question, call ask_direct() and print the query and response.
# Save the results in a list or dict so we can compare later.

SAMPLE_QUERIES = [
    "What is chain-of-thought prompting?",
    "What is retrieval augmented generation?",
    "What is sqrt(98765)?",
    "When was the Toolformer paper published?",
]

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 2: Retrieval-Augmented Generation

RAG extends the model by connecting it to an external **document corpus**. Instead of relying solely on parametric knowledge, we:

1. **Chunk** documents into manageable pieces
2. **Embed** each chunk into a vector representation
3. **Index** the vectors for fast similarity search
4. **Retrieve** the most relevant chunks for a given query
5. **Generate** an answer using the retrieved context

```
documents --> chunking --> embeddings --> vector index
                                              |
user query --> embed query --> search index --> retrieved chunks --> model --> answer
```

We have a small corpus of documents in the `docs/` directory covering topics like chain-of-thought prompting, RAG, Toolformer, and more.

In [ ]:
# =============================================================================
# PROVIDED: Load documents from the corpus
# =============================================================================

def load_documents(doc_dir="docs"):
    """Load all .txt files from the document directory.

    Returns:
        list of dicts, each with:
            'filename': name of the file
            'content': full text content
    """
    documents = []
    for filename in sorted(os.listdir(doc_dir)):
        if filename.endswith(".txt"):
            filepath = os.path.join(doc_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()
            documents.append({"filename": filename, "content": content})
    return documents


documents = load_documents()
print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  {doc['filename']:40s} ({len(doc['content']):,} chars)")

**TODO 2.1:** Write a function `chunk_documents(documents, chunk_size=500, overlap=50)` that splits each document into overlapping text chunks.

Chunking is necessary because:
- Embedding models have limited input length
- Smaller chunks allow more precise retrieval
- Overlap ensures we don't lose information at chunk boundaries

In [ ]:
# TODO 2.1: Implement document chunking (~15-20 lines)
#
# For each document, split the content into chunks of `chunk_size` characters
# with `overlap` characters of overlap between consecutive chunks.
#
# Hint: Use a sliding window approach:
#   start = 0
#   while start < len(text):
#       chunk = text[start:start + chunk_size]
#       start += chunk_size - overlap
#
# Return a list of dicts, each with:
#   'text': the chunk text
#   'source': the filename it came from
#   'chunk_index': index of this chunk within its document

def chunk_documents(documents, chunk_size=500, overlap=50):
    """Split documents into overlapping text chunks."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


chunks = chunk_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nFirst chunk:")
print(f"  Source: {chunks[0]['source']}")
print(f"  Text: {chunks[0]['text'][:200]}...")

**TODO 2.2:** Build a vector index using `sentence-transformers` and FAISS.

- [sentence-transformers](https://www.sbert.net/) provides pre-trained embedding models that convert text into dense vectors.
- [FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search) provides efficient similarity search over those vectors.

We use the `all-MiniLM-L6-v2` model, which produces 384-dimensional embeddings and is fast to run on CPU.

In [ ]:
# TODO 2.2: Build a FAISS index from chunks (~10-15 lines)
#
# Steps:
#   1. Load the sentence-transformers model:
#        from sentence_transformers import SentenceTransformer
#        embed_model = SentenceTransformer('all-MiniLM-L6-v2')
#
#   2. Encode all chunk texts into embeddings:
#        embeddings = embed_model.encode([c['text'] for c in chunks])
#
#   3. Build a FAISS index:
#        import faiss
#        import numpy as np
#        dimension = embeddings.shape[1]
#        index = faiss.IndexFlatL2(dimension)
#        index.add(np.array(embeddings).astype('float32'))
#
# Return: (index, chunks, embed_model)

def build_index(chunks):
    """Build a FAISS vector index from text chunks."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


index, chunks, embed_model = build_index(chunks)
print(f"Index built with {index.ntotal} vectors of dimension {index.d}")

**TODO 2.3:** Write a function `retrieve(query, index, chunks, model, k=3)` that:
1. Embeds the query using the same embedding model
2. Searches the FAISS index for the k nearest chunks
3. Returns the top-k chunks with their distances

In [ ]:
# TODO 2.3: Implement retrieval (~10-15 lines)
#
# Hint:
#   query_embedding = model.encode([query]).astype('float32')
#   distances, indices = index.search(query_embedding, k)
#
# Return a list of dicts, each with:
#   'text': the chunk text
#   'source': the filename
#   'distance': the L2 distance

import numpy as np

def retrieve(query, index, chunks, model, k=3):
    """Retrieve the top-k most relevant chunks for a query."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test retrieval
results = retrieve("What is chain-of-thought prompting?", index, chunks, embed_model)
print("Retrieved chunks:")
for i, r in enumerate(results):
    print(f"  {i+1}. [{r['source']}] (dist={r['distance']:.3f})")
    print(f"     {r['text'][:100]}...")

**TODO 2.4:** Write a function `ask_with_retrieval(query, index, chunks, model)` that:
1. Retrieves relevant chunks using `retrieve()`
2. Constructs a prompt with the retrieved context
3. Calls the language model with the augmented prompt
4. Returns the response and metadata

In [ ]:
# TODO 2.4: Implement RAG answering (~15-20 lines)
#
# Hint: Build a prompt that includes the retrieved chunks as context:
#
#   "Based on the following context, answer the question.
#
#    Context:
#    [chunk 1 text]
#    [chunk 2 text]
#    ...
#
#    Question: {query}"
#
# Return a dict with keys:
#   'response': str (the model's answer)
#   'retrieved_chunks': list of retrieved chunk dicts
#   'total_tokens': int

def ask_with_retrieval(query, index, chunks, model):
    """Answer a question using retrieval-augmented generation."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
test = ask_with_retrieval("What is chain-of-thought prompting?", index, chunks, embed_model)
print(f"Response: {test['response'][:300]}...")
print(f"Tokens: {test['total_tokens']}")
print(f"Sources: {[c['source'] for c in test['retrieved_chunks']]}")

**TODO 2.5:** Test RAG on the same sample questions from Section 1 and compare the results.

In [ ]:
# TODO 2.5: Compare direct vs RAG answers (~15-20 lines)
#
# For each question in SAMPLE_QUERIES:
#   1. Call ask_with_retrieval()
#   2. Print the query, direct answer (from Section 1), and RAG answer side by side
#
# Note which questions benefit from retrieval and which don't.
# (e.g., math questions won't benefit since our corpus doesn't contain math answers)

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 3: Tools

Retrieval connects the model to static knowledge. **Tools** connect it to dynamic capabilities: computation, search, APIs, databases, and more.

We will implement two tools:
- **Calculator**: evaluate math expressions precisely
- **Internet search**: query the web for current information

These address two weaknesses we saw in Section 1:
- Models struggle with precise arithmetic (sqrt(98765))
- Models can't access information after their training cutoff

**TODO 3.1:** Write a `calculator(expression)` function that safely evaluates math expressions.

Safety is important: we don't want to execute arbitrary Python code. Use a restricted `eval` that only allows math operations.

In [ ]:
# TODO 3.1: Implement a safe calculator tool (~10-15 lines)
#
# Hint: Use eval() with restricted globals to allow only math functions:
#
#   import math
#   allowed = {"__builtins__": {}, "math": math, "sqrt": math.sqrt,
#              "sin": math.sin, "cos": math.cos, "pi": math.pi, ...}
#   result = eval(expression, allowed)
#
# Wrap in a try/except to handle invalid expressions gracefully.
#
# Return: str (the result as a string, or an error message)

import math

def calculator(expression):
    """Safely evaluate a math expression and return the result as a string."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test
print(calculator("sqrt(98765)"))
print(calculator("15 * 37"))
print(calculator("2 ** 10"))
print(calculator("sin(pi / 4)"))

**TODO 3.2:** Write an `internet_search(query, k=3)` function that searches the web using DuckDuckGo.

In [ ]:
# TODO 3.2: Implement internet search tool (~10-15 lines)
#
# Hint:
#   from duckduckgo_search import DDGS
#
#   DuckDuckGo results can be flaky. Use a retry loop:
#   for attempt in range(retries + 1):
#       try:
#           with DDGS() as ddgs:
#               for r in ddgs.text(query, max_results=k):
#                   # r has keys: 'title', 'body', 'href'
#           if results:
#               return results
#           time.sleep(1)  # wait before retry
#       except Exception:
#           time.sleep(1)
#
# Return: list of dicts, each with 'title', 'snippet', 'url'
# If the search fails after all retries, return an empty list.

from duckduckgo_search import DDGS

def internet_search(query, k=3, retries=2):
    """Search the internet using DuckDuckGo and return top results."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test
results = internet_search("Who won the 2024 Super Bowl?")
for r in results:
    print(f"  {r['title']}")
    print(f"  {r['snippet'][:100]}...")
    print()

---

## Section 4: Function Calling

So far, we've been manually deciding when to use each tool. **Function calling** lets the model itself decide which tool to use and what arguments to pass.

The flow:
```
user query --> model (with tool definitions) --> tool_call(name, args)
                                                       |
                                                 execute tool
                                                       |
                                                 tool result --> model --> final answer
```

We need:
1. **Tool schemas**: structured descriptions of each tool (name, description, parameters)
2. **A dispatcher**: given a tool name and arguments, call the right function
3. **A conversation loop**: send query -> get tool call -> execute -> send result back

We use [Pydantic](https://docs.pydantic.dev/) to define clean, validated schemas.

**TODO 4.1:** Define Pydantic models for the calculator and internet_search tool parameters, then convert them to OpenAI-compatible tool definitions.

In [ ]:
# TODO 4.1: Define tool schemas (~20-30 lines)
#
# Step 1: Define Pydantic models for the parameters of each tool.
#
#   from pydantic import BaseModel, Field
#
#   class CalculatorInput(BaseModel):
#       expression: str = Field(description="A math expression to evaluate, e.g. 'sqrt(144)'")
#
#   class InternetSearchInput(BaseModel):
#       query: str = Field(description="The search query")
#       k: int = Field(default=3, description="Number of results to return")
#
# Step 2: Create OpenAI-compatible tool definitions (list of dicts).
#   Each tool definition should have this structure:
#   {
#       "type": "function",
#       "function": {
#           "name": "calculator",
#           "description": "Evaluate a math expression. Use for arithmetic, square roots, etc.",
#           "parameters": CalculatorInput.model_json_schema()
#       }
#   }
#
# Store the list in a variable called `tools`.

from pydantic import BaseModel, Field

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")


# Print the tool definitions
print(json.dumps(tools, indent=2))

In [ ]:
# =============================================================================
# PROVIDED: Tool dispatcher
# =============================================================================

def execute_tool_call(tool_name, arguments):
    """Execute a tool call by name with the given arguments.

    Args:
        tool_name: Name of the tool ('calculator' or 'internet_search')
        arguments: dict of arguments to pass to the tool

    Returns:
        str: The tool's output as a string
    """
    if tool_name == "calculator":
        return calculator(arguments["expression"])
    elif tool_name == "internet_search":
        results = internet_search(arguments["query"], arguments.get("k", 3))
        return json.dumps(results, indent=2)
    else:
        return f"Unknown tool: {tool_name}"


print("execute_tool_call() dispatcher defined.")

**TODO 4.2:** Write a function `ask_with_tools(query, tools)` that:
1. Sends the query to the model along with tool definitions
2. If the model responds with tool calls, executes them
3. Sends the tool results back to the model
4. Returns the final answer

In [ ]:
# TODO 4.2: Implement function calling loop (~30-40 lines)
#
# Hint: The OpenAI-compatible API flow for function calling:
#
#   1. Send the initial request with tools:
#      response = client.chat.completions.create(
#          model=MODEL,
#          messages=[{"role": "user", "content": query}],
#          tools=tools,
#          temperature=0.0,
#      )
#
#   2. Check if the model wants to call a tool:
#      message = response.choices[0].message
#      if message.tool_calls:
#          # Extract tool name and arguments
#          tool_call = message.tool_calls[0]
#          tool_name = tool_call.function.name
#          arguments = json.loads(tool_call.function.arguments)
#
#   3. Execute the tool and send the result back:
#      tool_result = execute_tool_call(tool_name, arguments)
#      # Build new messages list with the tool result
#      messages = [
#          {"role": "user", "content": query},
#          message,  # the assistant's message with tool_calls
#          {"role": "tool", "tool_call_id": tool_call.id, "content": tool_result}
#      ]
#      # Get final response
#      final = client.chat.completions.create(model=MODEL, messages=messages)
#
# Return a dict with keys:
#   'response': str (final answer)
#   'tool_called': str or None (name of tool called)
#   'tool_args': dict or None (arguments passed)
#   'tool_result': str or None (tool output)
#   'total_tokens': int

def ask_with_tools(query, tools):
    """Answer a question using function calling with tools."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test with a math question
test = ask_with_tools("What is sqrt(98765)?", tools)
print(f"Tool called: {test['tool_called']}")
print(f"Tool args: {test['tool_args']}")
print(f"Tool result: {test['tool_result']}")
print(f"Response: {test['response']}")

In [ ]:
# Test with a search question
test = ask_with_tools("Who won the 2024 Super Bowl?", tools)
print(f"Tool called: {test['tool_called']}")
print(f"Response: {test['response']}")

---

## Section 5: Agent Router

We now have four capabilities:
1. **Direct**: answer from parametric knowledge
2. **Retrieval**: answer using our document corpus
3. **Calculator**: compute math expressions
4. **Internet search**: look up current information

An **agent** decides which capability to use for each query. This is the **routing** problem: given a user query, which path should we take?

We'll use the model itself to make this decision by giving it a system prompt that describes the available routes.

**TODO 5.1:** Write a function `route_query(query)` that asks the model to classify the query into one of the available routes.

In [ ]:
# TODO 5.1: Implement query routing (~15-25 lines)
#
# Write a system prompt that describes the available routes:
#   - "direct": for simple questions the model can answer from general knowledge
#   - "retrieval": for questions about specific AI/ML topics in our corpus
#     (chain-of-thought, RAG, Toolformer, prompt engineering, function calling,
#      language model agents)
#   - "calculator": for math calculations
#   - "internet_search": for current events or facts that need web search
#
# Ask the model to respond with ONLY the route name (one word).
# Parse the response to extract one of the valid route names.
# If the response doesn't match, default to "direct".
#
# Return: str (one of 'direct', 'retrieval', 'calculator', 'internet_search')

VALID_ROUTES = ["direct", "retrieval", "calculator", "internet_search"]

def route_query(query):
    """Classify a query into one of the available routes."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test routing
test_queries = [
    "What is 15 * 37?",
    "What is chain-of-thought prompting?",
    "Who won the 2024 Super Bowl?",
    "What color is the sky?",
]
for q in test_queries:
    route = route_query(q)
    print(f"  {q:50s} -> {route}")

**TODO 5.2:** Write a function `agent_answer(query, index, chunks, embed_model, tools)` that:
1. Routes the query
2. Executes the chosen path
3. Returns the answer and metadata

In [ ]:
# TODO 5.2: Implement the full agent pipeline (~25-35 lines)
#
# Based on the route, call the appropriate function:
#   - "direct": call ask_direct(query)
#   - "retrieval": call ask_with_retrieval(query, index, chunks, embed_model)
#   - "calculator": extract the expression and call calculator(), then
#                    use ask_direct() with the result included in the prompt
#   - "internet_search": call ask_with_tools(query, tools)
#
# Return a dict with keys:
#   'query': the original query
#   'route': the chosen route
#   'response': the final answer string
#   'total_tokens': int
#   'metadata': dict with any additional info (e.g., retrieved chunks, tool results)

def agent_answer(query, index, chunks, embed_model, tools):
    """Full agent pipeline: route, execute, and return answer."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test the agent
test_queries = [
    "What is 15 * 37?",
    "What is chain-of-thought prompting?",
    "Who won the 2024 Super Bowl?",
    "What color is the sky?",
]
for q in test_queries:
    result = agent_answer(q, index, chunks, embed_model, tools)
    print(f"Query: {q}")
    print(f"Route: {result['route']}")
    print(f"Answer: {result['response'][:200]}")
    print()

---

## Section 6: Trace Logging

As systems get more complex, it becomes essential to **log what happens** at each step. A **trace** records the full sequence of decisions and actions, making it possible to debug failures and understand behavior.

Each trace step records:
- **Step type**: what kind of action (routing, retrieval, tool_call, generation, etc.)
- **Input**: what went into this step
- **Output**: what came out
- **Metadata**: any additional context (tokens used, sources, etc.)

**TODO 6.1:** Create a `Trace` class that logs agent steps.

In [ ]:
# TODO 6.1: Implement the Trace class (~25-35 lines)
#
# The Trace class should:
#   - Store a query (str) and a list of steps (list of dicts)
#   - Store a route (str, initially None) and final_answer (str, initially None)
#   - Have an add_step(step_type, input, output, metadata=None) method
#     that appends a step dict to the list
#   - Have a to_dict() method that returns the full trace as a dict
#     (including query, route, final_answer, steps, and total_time)
#   - Have a save(filename) method that saves the trace as JSON
#     to a 'traces/' directory (create it if it doesn't exist)
#
# Each step dict should have keys:
#   'step_type', 'input', 'output', 'metadata', 'timestamp'
#
# Hint: Use time.time() for timestamps.

class Trace:
    """Log agent execution steps for debugging and analysis."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Quick test
t = Trace("test query")
t.add_step("routing", "test query", "direct")
t.add_step("generation", "test query", "test answer", {"tokens": 42})
print(json.dumps(t.to_dict(), indent=2))

**TODO 6.2:** Update the agent to use trace logging. Write `agent_answer_traced(query, index, chunks, embed_model, tools)` that behaves like `agent_answer` but also populates and returns a `Trace` object.

In [ ]:
# TODO 6.2: Implement traced agent (~30-45 lines)
#
# This is similar to agent_answer, but at each step, log to a Trace:
#   - Step 1: routing (input=query, output=route)
#   - Step 2: depends on route:
#     * retrieval: log retrieval step (input=query, output=retrieved chunks)
#     * tool_call: log tool call step (input=tool args, output=tool result)
#   - Step 3: generation (input=prompt, output=response)
#
# Return a dict with the same keys as agent_answer plus:
#   'trace': the Trace object

def agent_answer_traced(query, index, chunks, embed_model, tools):
    """Agent pipeline with trace logging."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Test and save a trace
result = agent_answer_traced("What is sqrt(144)?", index, chunks, embed_model, tools)
print(f"Route: {result['route']}")
print(f"Answer: {result['response']}")
result['trace'].save("test_trace.json")
print("Trace saved!")

In [ ]:
# Run a few more queries and save traces
trace_queries = [
    "What is chain-of-thought prompting?",
    "What is 15 * 37?",
    "Who won the 2024 Super Bowl?",
    "What color is the sky?",
]

for q in trace_queries:
    result = agent_answer_traced(q, index, chunks, embed_model, tools)
    safe_name = q.replace(" ", "_").replace("?", "")[:40] + ".json"
    result['trace'].save(safe_name)
    print(f"  [{result['route']:15s}] {q} -> saved trace")

---

## Section 7: Trace Visualization

Traces are most useful when you can **see** them. We provide a simple Streamlit viewer (`trace_viewer.py`) that you can run with:

```bash
streamlit run trace_viewer.py
```

The viewer loads trace JSON files from the `traces/` directory and displays the query, route, and each step in order.

For now, let's build a simple text-based viewer right here in the notebook.

**TODO 7.1:** Load a saved trace JSON file and print a formatted text summary.

In [ ]:
# TODO 7.1: Build a text-based trace viewer (~15-25 lines)
#
# Write a function display_trace(filepath) that:
#   1. Loads the trace JSON from the file
#   2. Prints a formatted summary:
#      - Query
#      - Route
#      - For each step:
#        * Step number and type
#        * Input (truncated to 200 chars)
#        * Output (truncated to 200 chars)
#        * Any metadata
#
# Use clear formatting with separators between steps.

def display_trace(filepath):
    """Load and display a trace file in a readable format."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete this TODO")


# Display one of our saved traces
display_trace("traces/test_trace.json")

---

## Section 8: Tool Description Experiments

The descriptions we give tools significantly affect how and when the model chooses to use them. In this section, we experiment with **minimal** vs **expanded** tool descriptions to see how they change routing behavior.

This is an important lesson: **prompt engineering applies to tool descriptions too, not just user prompts.**

**TODO 8.1:** Create two sets of tool descriptions: minimal and expanded.

In [ ]:
# TODO 8.1: Create minimal and expanded tool descriptions (~20-30 lines)
#
# Minimal descriptions:
#   calculator: "Evaluate math expressions."
#   internet_search: "Search the internet."
#
# Expanded descriptions:
#   calculator: "Use for arithmetic calculations such as addition, subtraction,
#                multiplication, division, square roots, trigonometry, and exponents.
#                Input should be a valid Python math expression."
#   internet_search: "Use for looking up factual information, recent events,
#                     current data, or anything that requires up-to-date web information.
#                     Input should be a search query string."
#
# Create two lists of tool definitions: `tools_minimal` and `tools_expanded`
# Each should follow the same OpenAI tool definition format as in Section 4.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")


print("Minimal tools:")
for t in tools_minimal:
    print(f"  {t['function']['name']}: {t['function']['description']}")
print("\nExpanded tools:")
for t in tools_expanded:
    print(f"  {t['function']['name']}: {t['function']['description']}")

**TODO 8.2:** Run the same set of test queries with each description set. Record and compare routing decisions.

In [ ]:
# TODO 8.2: Compare routing with minimal vs expanded descriptions (~25-35 lines)
#
# Use these test queries:
DESCRIPTION_TEST_QUERIES = [
    "What is 15 * 37?",
    "What is sqrt(144)?",
    "Who won the 2024 Super Bowl?",
    "What is the current price of Bitcoin?",
    "What is 2 to the power of 10?",
    "What is the latest news about AI?",
]

# For each query, call ask_with_tools() with tools_minimal and tools_expanded.
# Record which tool was called (or None) for each.
#
# Print a comparison table:
#   Query | Minimal (tool called) | Expanded (tool called) | Same?
#
# Discuss: which description set leads to better tool selection?

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Section 9: Evaluating Agent Decisions

How well does our agent route queries to the right capability? Let's measure this systematically with a set of test queries that have **expected routes**.

In [ ]:
# =============================================================================
# PROVIDED: Evaluation queries with expected routes
# =============================================================================

EVAL_QUERIES = [
    {"query": "What is 15 * 37?", "expected_route": "calculator"},
    {"query": "What is chain-of-thought prompting?", "expected_route": "retrieval"},
    {"query": "Who won the 2024 Super Bowl?", "expected_route": "internet_search"},
    {"query": "What color is the sky?", "expected_route": "direct"},
    {"query": "What is sqrt(144)?", "expected_route": "calculator"},
    {"query": "When was the Toolformer paper published?", "expected_route": "retrieval"},
    {"query": "What is the current price of Bitcoin?", "expected_route": "internet_search"},
    {"query": "Explain what RAG stands for", "expected_route": "direct"},
]

print(f"Evaluation set: {len(EVAL_QUERIES)} queries")
for eq in EVAL_QUERIES:
    print(f"  [{eq['expected_route']:15s}] {eq['query']}")

**TODO 9.1:** Run each eval query through the agent and record the actual route and answer.

In [ ]:
# TODO 9.1: Run evaluation queries (~15-20 lines)
#
# For each query in EVAL_QUERIES:
#   1. Call agent_answer_traced() (or agent_answer())
#   2. Record the actual route and response
#   3. Compare actual route to expected route
#
# Store results in a list of dicts with keys:
#   'query', 'expected_route', 'actual_route', 'correct', 'response'
#
# Print a results table.

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

**TODO 9.2:** Compute and display evaluation metrics.

In [ ]:
# TODO 9.2: Compute evaluation metrics (~20-30 lines)
#
# Compute and display:
#   1. Overall route accuracy (correct routes / total queries)
#   2. A confusion-style table showing:
#        Expected route | Actual route | Count
#      for all (expected, actual) pairs
#   3. List the failure cases (queries where the route was wrong)
#      with the expected and actual routes
#
# Hint: Use collections.Counter for counting route pairs.

from collections import Counter

# YOUR CODE HERE
raise NotImplementedError("Complete this TODO")

---

## Reflection Questions

Answer the following questions based on your experiments.

**Q1: When did retrieval improve answers?**

Compare the direct answers (Section 1) with the RAG answers (Section 2). For which questions did retrieval provide better, more specific, or more accurate answers? Why?

*Write your answer here...*

**Q2: When did tool calls help?**

For which questions did tools provide better answers than the model alone? What types of tasks benefit most from tool access?

*Write your answer here...*

**Q3: When did the agent choose the wrong route?**

Look at the evaluation results from Section 9. Which queries were misrouted? What might explain those errors?

*Write your answer here...*

**Q4: How did tool descriptions affect tool selection?**

Compare the results from Section 8. Did the minimal or expanded descriptions lead to better tool selection? What does this tell us about prompt engineering for tools?

*Write your answer here...*

**Q5: What information in the trace helped debug failures?**

Look at the traces you saved in Section 6. When the agent gave a wrong answer, what information in the trace helped you understand why? What additional information would you want to log?

*Write your answer here...*

---

## Congratulations!

You have completed the retrieval, tools, and agent interfaces lab. Here is a summary of what you built:

1. **Direct answering**: Baseline using only the model's parametric knowledge.
2. **RAG pipeline**: Document chunking, embedding, vector search, and context-augmented generation.
3. **Tool implementation**: A safe calculator and internet search tool.
4. **Function calling**: Structured tool invocation with Pydantic schemas and the OpenAI tool calling API.
5. **Agent router**: A system that decides which capability to use for each query.
6. **Trace logging**: Step-by-step execution logs for debugging and analysis.
7. **Tool description experiments**: Demonstrated how tool descriptions affect model behavior.
8. **Evaluation**: Measured routing accuracy with expected-route annotations.

The big takeaway: **modern AI systems are not just models. They are systems built around models that connect to knowledge (retrieval), capabilities (tools), and decision-making (agents).** Understanding how to build, debug, and evaluate these systems is essential for working with AI in practice.

---

### Instructor Notes

**Key concepts reinforced:**
- Retrieval-augmented generation (RAG) and when it helps vs. direct answering
- Tool use as an extension of model capabilities
- Structured function calling with schemas
- Agent routing and the decision of which capability to invoke
- Trace logging for debugging complex AI systems
- The importance of tool descriptions in guiding model behavior

**Expected results (approximate, will vary by model):**
- RAG should improve answers for corpus-specific questions (chain-of-thought, Toolformer, etc.)
- Calculator should precisely answer math questions that the model approximates
- Internet search should answer current-events questions
- Routing accuracy: typically 6-8/8 correct with a good routing prompt
- Expanded tool descriptions generally lead to better tool selection

**Common student issues:**
- FAISS installation: if `faiss-cpu` fails, try `pip install faiss-cpu --no-cache-dir`
- DuckDuckGo rate limits: add `time.sleep(1)` between search calls if needed
- Tool calling format: the exact API format varies slightly by model; Gemini Flash works well
- Routing ambiguity: some queries are genuinely ambiguous (e.g., "Explain what RAG stands for" could be direct or retrieval)

**Discussion points for lecture tie-in:**
- How do production RAG systems differ from our simple implementation? (reranking, hybrid search, metadata filtering)
- How do agents like ChatGPT plugins or Claude tool use work under the hood?
- What are the risks of giving models access to tools? (prompt injection, unintended actions)
- How does trace logging relate to observability in production ML systems?